# rocket representation

this notebook builds the `Rocket` class — the interface through which rocket designs are constructed, inspected, and validated.

## context

goal 0 gave us two things: a parts library (`data/parts_library.json`, 358 stock KSP1 parts) and a structural validator (`src/structure.py`) that checks whether a rocket dict is well-formed. the validator expects a dict of this shape:

```json
{
  "parts": [
    {"id": "pod_0",  "type": "mk1-3pod",     "parent": null},
    {"id": "tank_0", "type": "fuelTank",     "parent": "pod_0",  "attach_node": "bottom"},
    {"id": "eng_0",  "type": "liquidEngine", "parent": "tank_0", "attach_node": "bottom"}
  ],
  "stages": {"eng_0": 0}
}
```

so far, those dicts have been written by hand. that's fine for testing, but it's not how a generator or search algorithm will produce designs. we need a programmatic interface.

## what we're building

the `Rocket` class lets you construct a rocket dict through controlled method calls:

```python
r = Rocket(parts_library)
r.add_part("pod_0",  "mk1-3pod",     parent=None)
r.add_part("tank_0", "fuelTank",     parent="pod_0",  attach_node="bottom")
r.add_part("eng_0",  "liquidEngine", parent="tank_0", attach_node="bottom")
r.set_stage("eng_0", 0)

r.to_dict()    # produces the rocket dict for the validator
r.validate()   # runs validate_rocket and returns the result
```

the class is the construction interface. its output (`to_dict()`) is what everything downstream consumes — the validator, the simulator, eventually the search algorithm.

## what we're not building (yet)

we're not committing to a training representation here. whether designs eventually get represented as action sequences (for an autoregressive model), graphs (for a GNN), or fixed-length vectors (for evolutionary search) is a decision for later, once we know what kind of model we're building. the `Rocket` class is architecture-agnostic — it's just a clean way to construct and inspect designs.

## plan

1. sketch the class interface — what methods do we need, what should they return?
2. implement the internal state — how does the class store parts and stages?
3. implement `add_part` with validation
4. implement `set_stage` with validation
5. implement `to_dict` and `validate`
6. implement `__repr__` for easy inspection
7. test against `data/toy_rocket.json` and some invalid cases
8. extract to `src/rocket.py`

In [ ]:
from src.structure import check_part_call, validate_rocket
class Rocket:
    def __init__(self, parts_library):
        self.parts_library = parts_library
        self.parts = []
        self.stages = {}

    def __repr__(self):
        return f"Rocket {len(self.parts)} parts, {len(self.stages)} staged"

    def add_part(self, id, part_type, parent, attach_node = None):
        exists = check_part_call(part_type, self.parts_library)
        if not exists:
            raise ValueError(f"unknown part type: {part_type}")

        existing_ids = {p['id'] for p in self.parts}

        if id in existing_ids:
            raise ValueError(f"{id} already in structure")

        if parent is not None:
            if parent not in existing_ids:
                raise ValueError(f"{parent} not in structure")

        part = {'id': id, 'type': part_type, 'parent': parent}
        if attach_node is not None:
            part['attach_node'] = attach_node

        self.parts.append(part)

        return self #leaving this in in case i want method chaining later

    def set_stage(self, part_id, stage):
        existing_ids = {p['id'] for p in self.parts}
        if part_id not in existing_ids:
            raise ValueError(f'{part_id} not in structure')

        if not isinstance(stage, int) or stage < 0:
            raise ValueError('invalid stage value')

        self.stages[part_id] = stage

        return self

    def to_dict(self):
        out_dict = {'parts': self.parts,
                    'stages': self.stages}

        return out_dict

    def validate(self, verbose = False):
        return validate_rocket(self.to_dict(), self.parts_library, verbose = verbose)





In [2]:
import json
from pathlib import Path
with open(Path('../data/parts_library.json')) as f:
    parts_library = json.load(f)

parts_by_name = {p['name']: p for p in parts_library}

with open(Path('../data/toy_rocket.json')) as f:
    toy = json.load(f)

print(json.dumps(toy, indent = 2))

{
  "parts": [
    {
      "id": "pod_0",
      "type": "mk1-3pod",
      "parent": null
    },
    {
      "id": "tank_0",
      "type": "fuelTank",
      "parent": "pod_0",
      "attach_node": "bottom"
    },
    {
      "id": "eng_0",
      "type": "liquidEngine",
      "parent": "tank_0",
      "attach_node": "bottom"
    }
  ],
  "stages": {
    "eng_0": 0
  }
}


In [ ]:

r = Rocket(parts_by_name)
r.add_part('pod_0', 'mk1-3pod', parent=None)
r.add_part('tank_0', 'fuelTank', parent = 'pod_0', attach_node = 'bottom')
r.add_part('eng_0','liquidEngine', parent = 'tank_0', attach_node = 'bottom')
r.set_stage('eng_0', 0)

print(r.to_dict())
print(r.validate(verbose = True))
print(r.__repr__())

In [ ]:
r2 = Rocket(parts_by_name)
r2.add_part('pod_0', 'mk1-3pod', parent=None)
r2.add_part('pod_0', 'mk1-3pod', parent=None)  # should raise ValueError

In [ ]:
r3 = Rocket(parts_by_name)
r3.add_part('tank_0', 'fuelTank', parent='ghost_pod', attach_node='bottom')  # should raise ValueError

In [3]:
from src.rocket import Rocket

r = Rocket(parts_by_name)
r.add_part('pod_0', 'mk1-3pod', parent=None)
r.add_part('tank_0', 'fuelTank', parent = 'pod_0', attach_node = 'bottom')
r.add_part('eng_0','liquidEngine', parent = 'tank_0', attach_node = 'bottom')
r.set_stage('eng_0', 0)

print(r.to_dict())
print(r.validate(verbose = True))
print(r.__repr__())

{'parts': [{'id': 'pod_0', 'type': 'mk1-3pod', 'parent': None}, {'id': 'tank_0', 'type': 'fuelTank', 'parent': 'pod_0', 'attach_node': 'bottom'}, {'id': 'eng_0', 'type': 'liquidEngine', 'parent': 'tank_0', 'attach_node': 'bottom'}], 'stages': {'eng_0': 0}}
True
Rocket 3 parts, 1 staged
